# PQC experiments — GitHub → Colab → all results

This companion to `PQC_MIMO_Simulations.ipynb` runs the same experiment modules and adds the full ordered execution and reporting workflow. **Upload this notebook to Colab, select a runtime, and Run all.** It clones `DanielGrahamB/pqc-research` from `rev1` by default. Push the complete refactored project to that branch first. No embedded source snapshot is used here.

**Run all runs every experiment at pilot scale by default.** Experiment 7 has separate K and M subgroups, so the ten research families produce eleven output folders. Every completed experiment shows numerical tables, BER/BLER curves, annotated rate heatmaps (so overlapping curves stay visible), goodput and processing time. Experiment 10 adds component latency, key-storage and power-expansion figures. PNGs and CSVs are saved automatically.

Set `MODE="view_saved"` to display existing `results/pilot` folders without rerunning them. Those folders are saved outputs, not executable experiments; the shared `paper_groups` function defines what to simulate. You can also point the saved-results path at an extracted Colab download. Missing saved groups are reported explicitly.

Default pilot runs are exploratory. Flat BER near .5 and BLER=1 indicate a poor operating range for that configuration; completing the workflow does not establish a GGH/LDPC/subband gain or cryptographic security. The original notebook documents signal domains, assumptions and fairness limitations.


## 1. Clone the project

Set the branch/tag below to where you push. Existing checkouts are reused without pulling or overwriting edits. To load a newer pushed revision, use a fresh Colab runtime. A local run uses the current repository directory and does not clone. No credentials are embedded; the default assumes the repository is accessible to the runtime.


In [ ]:
# GitHub source: push the complete refactored project to this branch first.
REPO_URL = "https://github.com/DanielGrahamB/pqc-research.git"
GIT_REF = "rev1"  # Existing branch or tag; change if you push elsewhere.

import os
import sys
import subprocess
from pathlib import Path
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
if IN_COLAB:
    CHECKOUT = Path("/content/pqc-research")
    if not CHECKOUT.exists():
        subprocess.check_call(["git", "clone", "--branch", GIT_REF,
                               "--single-branch", REPO_URL, str(CHECKOUT)])
    else:
        if not (CHECKOUT / ".git").is_dir():
            raise RuntimeError(f"{CHECKOUT} exists but is not a Git checkout. Use a fresh runtime.")
        origin = subprocess.check_output(["git", "-C", str(CHECKOUT), "remote", "get-url", "origin"], text=True).strip()
        if origin.removesuffix('.git') != REPO_URL.removesuffix('.git'):
            raise RuntimeError(f"Checkout belongs to a different repository: {origin}")
        head = subprocess.check_output(["git", "-C", str(CHECKOUT), "rev-parse", "HEAD"], text=True).strip()
        requested = subprocess.check_output(["git", "-C", str(CHECKOUT), "rev-parse", GIT_REF], text=True).strip()
        if head != requested:
            raise RuntimeError("Checkout differs from GIT_REF. Use a fresh runtime for another revision.")
        print("Reusing this checkout. To fetch newly pushed code, use a fresh runtime; local files are not overwritten.")
else:
    CHECKOUT = Path.cwd()

candidates = [CHECKOUT, CHECKOUT / "ps-layer-security"]
PROJECT_ROOT = next((p for p in candidates if (p / "pqc_experiments/runner.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Refactored modules not found. Push pqc_experiments/, metrics.py and requirements-colab.txt to the selected GitHub branch first.")
required = ["metrics.py", "requirements-colab.txt", "pqc_experiments/model.py", "pqc_experiments/validation.py"]
for name in required:
    if not (PROJECT_ROOT/name).is_file():
        raise FileNotFoundError(PROJECT_ROOT/name)
active = sys.modules.get("pqc_experiments")
if active is not None and Path(active.__file__).resolve().parent != PROJECT_ROOT.resolve()/"pqc_experiments":
    raise RuntimeError("Another project version is already imported. Restart the session before switching notebooks.")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
try:
    GIT_COMMIT = subprocess.check_output(["git", "-C", str(CHECKOUT), "rev-parse", "HEAD"], text=True, stderr=subprocess.DEVNULL).strip()
except subprocess.CalledProcessError:
    GIT_COMMIT = "local-unversioned"
print("Project:", PROJECT_ROOT)
print("Git commit:", GIT_COMMIT)


## 2. Dependencies

Sionna remains pinned to the project's tested API; a compatible Colab PyTorch installation is retained. If pip updates an imported numerical library, follow the displayed restart instruction, then Run all again. The clone remains on disk after a session restart. Local execution requires the project dependencies plus pandas.


In [ ]:
# Install only in Colab. On a local machine use your existing project environment.
import importlib
import importlib.metadata
if IN_COLAB:
    if sys.version_info < (3, 11):
        raise RuntimeError("Select a Colab runtime with Python 3.11 or newer.")
    from packaging.requirements import Requirement
    requirements = [Requirement(line) for line in (PROJECT_ROOT/"requirements-colab.txt").read_text().splitlines()
                    if line.strip() and not line.startswith("#")]
    requirements.append(Requirement("pandas>=2.2"))
    def version(name):
        try: return importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError: return None
    missing = [str(r) for r in requirements if version(r.name) is None or not r.specifier.contains(version(r.name))]
    if missing:
        loaded = {r.name: version(r.name) for r in requirements if r.name in sys.modules}
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                               "-r", str(PROJECT_ROOT/"requirements-colab.txt"), "pandas>=2.2"])
        importlib.invalidate_caches()
        changed = [name for name, before in loaded.items() if version(name) != before]
        if changed:
            raise RuntimeError("Updated imported libraries: " + ", ".join(changed) + ". Restart session, then Run all again. Checkout is preserved.")
print("Dependency setup complete.")


## 3. Run settings and workload preview

`pilot` uses five measured batches per configuration/SNR, with two warm-ups. `extended` is much larger and still needs sample-size and SNR refinement before paper claims. Configurations share the same key registry and deterministic seed policy. No Cartesian-product sweep is constructed.

New results go to a timestamped run directory. Re-executing cells reuses completed groups only if source, configurations, SNRs, packet counts and recorded software versions match. Incomplete groups restart in a new retry folder, preserving the partial attempt. Set a new `RUN_LABEL` when changing settings. To resume across a runtime replacement, enable Drive and use the same label and settings.

When viewing historical CPU results on a GPU runtime, the tables retain the saved measurements; the current GPU does not change those results. A Drive mount occurs only if explicitly enabled below.


In [ ]:
from dataclasses import asdict, replace
from datetime import datetime, timezone
import json
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import sionna
from IPython.display import display, Markdown
from sionna.phy import config as sionna_config
from pqc_experiments import ExperimentConfig, KeyRegistry
from pqc_experiments.runner import paper_groups, run_group, fingerprint
from pqc_experiments.validation import validate

MODE = "run"                 # "run" or "view_saved"
PROFILE = "pilot"            # "pilot" or "extended"
DEVICE = f"cuda:{torch.cuda.current_device()}" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 8
SHOW_CURVES = True
SHOW_HEATMAPS = True
SAVE_TO_DRIVE = False        # Colab asks for Drive authorization only if True.
SAVED_RESULTS_ROOT = PROJECT_ROOT / "results/pilot"
# You may point SAVED_RESULTS_ROOT at an extracted Colab download instead.

if MODE not in ("run", "view_saved") or PROFILE not in ("pilot", "extended"):
    raise ValueError("Choose the documented MODE and PROFILE values.")
assert DEVICE in sionna_config.available_devices
sionna_config.device = DEVICE
sionna_config.precision = "double"
sionna_config.seed = 20
torch.manual_seed(20)
torch.use_deterministic_algorithms(True)
reference = ExperimentConfig(batch_size=BATCH_SIZE, device=DEVICE)
registry = KeyRegistry()
groups = paper_groups(reference)

if PROFILE == "pilot":
    PACKETS, WARMUPS = 5, 2
    ebno_by_group = {name: [0,20,40,60,80] for name in groups}
    ebno_by_group["01_conventional_coding"] = [0,5,10,15,20]
else:
    PACKETS, WARMUPS = 100, 3
    ebno_by_group = {name: list(range(0,81,5)) for name in groups}
# Edit PACKETS or ebno_by_group here before running validation/experiments.

RUN_LABEL = globals().get("RUN_LABEL", datetime.now(timezone.utc).strftime("all_experiments_%Y%m%dT%H%M%SZ"))
if IN_COLAB and SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    output_base = Path("/content/drive/MyDrive/pqc-results")
else:
    output_base = Path("/content/pqc-results") if IN_COLAB else PROJECT_ROOT / "results"
OUTPUT_ROOT = output_base / RUN_LABEL
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
REPORT_ROOT = OUTPUT_ROOT / "reports"
REPORT_ROOT.mkdir(exist_ok=True)
print({"mode":MODE,"profile":PROFILE,"device":DEVICE,"torch":torch.__version__,"sionna":sionna.__version__})
print("New results/reports:", OUTPUT_ROOT)
print("Saved input results:", SAVED_RESULTS_ROOT)
plan = pd.DataFrame([{"group":name,"configurations":len(configs),"SNR_points":len(ebno_by_group[name]),
                      "batch_runs":len(configs)*len(ebno_by_group[name])*PACKETS} for name,configs in groups.items()])
display(plan)
print("Total measured batch runs:", int(plan.batch_runs.sum()), "; warm-ups and validation are additional.")


## 4. Validation before simulation

All 53 validation checks must pass before new experiments. Explicit `cuda:0`/`cpu` selection and the noise-tensor device fix are in the shared project code. This check also invalidates old source-hash validation records. In view-only mode the stored validation status is displayed per group.


In [ ]:
if MODE == "run":
    validation = validate(registry, output=str(OUTPUT_ROOT/"validation.json"), device=DEVICE)
    assert validation["passed"], "Validation failed; do not run experiments."
else:
    validation = None
    print("View-only mode: no simulations or new validation. Each report identifies the saved run's validation status.")
run_info = {"git_commit":GIT_COMMIT,"source_hash":fingerprint(),"mode":MODE,"profile":PROFILE,
            "device":DEVICE,"packets":PACKETS,"warmups":WARMUPS,"ebno_by_group":ebno_by_group,
            "reference":asdict(reference),"saved_input":str(SAVED_RESULTS_ROOT)}
(OUTPUT_ROOT/"notebook_run.json").write_text(json.dumps(run_info,indent=2))


## 5. Reporting and safe reruns

Execute the helper cell once. Rate plots use linear axes to show zero observed errors. Annotated heatmaps distinguish configurations even when curves coincide; C0, C1, etc. refer to the configuration-index table. Zero errors is not zero true probability. Timing is batch wall time, including Python overhead. The key-generation time is a one-time observation, not per-packet latency.

Full tables, including error counts, sample sizes, approximate confidence intervals and decryption diagnostics, are saved with every report. Coarse interpolated Eb/N0 thresholds are descriptive and may be unreliable with pilot samples. Failed groups are recorded and do not silently appear as completed; consult the final status table.


In [ ]:
# Reporting is separate from the communication model. All figures use saved CSVs.
STATUS = {}
REPORTS = {}
INDEX_PATH = OUTPUT_ROOT / "group_index.json"
GROUP_PATHS = json.loads(INDEX_PATH.read_text()) if INDEX_PATH.exists() else {}

LABEL_KEYS = ["scheme","scenario","K","M","modulation","receiver","csi","coderate","tile"]
def labels_for(df):
    varying = [k for k in LABEL_KEYS if k == "scheme" or df[k].nunique(dropna=False)>1]
    return {index:" | ".join(f"{k}={row[k]}" for k in varying)
            for index,row in df.groupby("config_index",sort=True).first().iterrows()}

def save_figure(fig,folder,name):
    fig.savefig(folder/f"{name}.png",dpi=180,bbox_inches="tight")
    display(fig)
    plt.close(fig)

def report_group(name,folder):
    df = pd.read_csv(folder/"summary.csv")
    manifest = json.loads((folder/"manifest.json").read_text())
    labels = labels_for(df)
    df["configuration"] = df.config_index.map(labels)
    target = REPORT_ROOT / name
    target.mkdir(exist_ok=True)
    display(Markdown(f"### {name}\nSource: `{folder}`"))
    print("Saved validation:", manifest.get("validation",{}).get("passed"),
          "| saved Sionna:",manifest.get("sionna"),"| saved device:",manifest["configs"][0].get("device"))
    cols = ["config_index","configuration","ebno_db","ber","bler","block_errors","num_blocks",
            "bler_ci_low","bler_ci_high","goodput","total_ms"]
    with pd.option_context("display.max_rows",None,"display.max_colwidth",100):
        display(df[cols].round(6))
    print("Heatmap row C0/C1/... refers to config_index in the table above.")
    df.to_csv(target/"annotated_summary.csv",index=False)
    if SHOW_CURVES:
        fig,axes=plt.subplots(2,2,figsize=(14,9),layout="constrained")
        palette=plt.get_cmap("tab10")
        for j,(index,curve) in enumerate(df.groupby("config_index",sort=True)):
            curve=curve.sort_values("ebno_db")
            for ax,metric,scale,title in zip(axes.flat,["ber","bler","goodput","total_ms"],
                                           [1,1,1e6,1],["BER (linear scale)","BLER (linear scale)","Goodput (Mbit/s)","Total processing (ms/batch)"]):
                ax.plot(curve.ebno_db,curve[metric]/scale,marker=["o","s","^","D"][j%4],
                        linestyle=["-","--",":","-."][j%4],color=palette(j%10),
                        markersize=max(4,10-j),markerfacecolor="none",linewidth=1.7,label=labels[index])
                ax.set(xlabel="Eb/N0 (dB)",ylabel=title,title=title)
                ax.grid(True,alpha=.25)
                if metric in ("ber","bler"): ax.set_ylim(-.025,1.06)
                else: ax.set_ylim(bottom=0)
        handles,text=axes[0,0].get_legend_handles_labels()
        fig.legend(handles,text,loc="outside lower center",fontsize=8,ncol=1)
        fig.suptitle(name)
        save_figure(fig,target,"curves")
    if SHOW_HEATMAPS:
        # Separate cells expose identical/overlapping curves without perturbing data.
        fig,axes=plt.subplots(1,2,figsize=(16,max(3.5,.65*len(labels)+2)),layout="constrained")
        for ax,metric in zip(axes,["ber","bler"]):
            table=df.pivot(index="config_index",columns="ebno_db",values=metric).sort_index()
            values=table.to_numpy()
            im=ax.imshow(values,aspect="auto",vmin=0,vmax=1,cmap="viridis")
            ax.set_xticks(range(len(table.columns)),[f"{v:g}" for v in table.columns])
            ax.set_yticks(range(len(table.index)),[f"C{i}" for i in table.index],fontsize=8)
            ax.set(xlabel="Eb/N0 (dB)",title=f"{metric.upper()} — exact sample rates")
            for y in range(values.shape[0]):
                for x in range(values.shape[1]):
                    value=values[y,x]
                    ax.text(x,y,f"{value:.4g}",ha="center",va="center",fontsize=9,
                            color="black" if value>.6 else "white")
            fig.colorbar(im,ax=ax,shrink=.7)
        save_figure(fig,target,"rate_tables")
    extra=[k for k in ["configuration","ebno_db","gamma_db","wrong_babai_coordinates",
                      "wrong_message_symbols","effective_spectral_efficiency","encrypt_ms","decrypt_ms",
                      "receiver_ms","ldpc_ms","keygen_ms","public_key_bytes","private_key_bytes",
                      "required_ebno_bler_1e2","threshold_status"] if k in df]
    with pd.option_context("display.max_rows",None,"display.max_colwidth",90): display(df[extra].round(5))
    if name == "10_overhead":
        high=df[df.ebno_db==df.ebno_db.max()].sort_values("config_index")
        fig,axes=plt.subplots(1,3,figsize=(16,5),layout="constrained")
        positions=np.arange(len(high));tick=[str(s).replace("_","\n") for s in high.scheme]
        for col in ["encrypt_ms","decrypt_ms","receiver_ms","ldpc_ms"]:
            axes[0].plot(positions,high[col],"o-",label=col)
        axes[0].set(ylabel="Mean ms/batch",title="Processing components");axes[0].legend(fontsize=8)
        for col in ["public_key_bytes","private_key_bytes"]:
            axes[1].plot(positions,high[col]/1024,"o-",label=col)
        axes[1].set(ylabel="KiB",title="Key tensor storage");axes[1].legend(fontsize=8)
        axes[2].bar(positions,high.gamma_db);axes[2].set(ylabel="dB",title="Ciphertext / integer-message power")
        for ax in axes:ax.set_xticks(positions,tick,fontsize=8);ax.grid(True,axis="y",alpha=.25)
        save_figure(fig,target,"overhead")
    if (df.bler==1).any():
        print("BLER=1 means every observed user block had an error; it does not mean every bit was wrong.")
    if (df.bler==0).any():
        print("Zero observed errors is a sample result. Inspect counts and confidence bounds before claiming BLER < 1e-2.")
    if ((df.ber>.45)&(df.ber<.55)&(df.bler==1)).any():
        print("Some configurations have BER near .5 and BLER=1. Treat those settings as an uninformative operating range for small reliability gains; diagnose power expansion/decryption sensitivity.")
    print("Threshold interpolation from pilot samples is descriptive, not a paper-quality estimate. Confidence intervals assume approximately independent blocks.")
    REPORTS[name]=df
    return df

def execute_and_report(name):
    import traceback
    try:
        if MODE == "view_saved":
            direct=SAVED_RESULTS_ROOT/name
            choices=[direct] if (direct/"summary.csv").is_file() else sorted(
                {p.parent for p in SAVED_RESULTS_ROOT.rglob("summary.csv") if p.parent.name==name})
            if not choices:
                STATUS[name]="missing saved group"
                print(f"{name}: no saved results under {SAVED_RESULTS_ROOT}; skipped.")
                return
            if len(choices)>1:
                raise RuntimeError(f"Multiple saved runs match {name}; set SAVED_RESULTS_ROOT to one run directory.")
            folder=choices[0]
        else:
            folder=Path(GROUP_PATHS.get(name,str(OUTPUT_ROOT/name)))
            if (folder/"summary.csv").exists():
                m=json.loads((folder/"manifest.json").read_text())
                expected=json.loads(json.dumps([asdict(c) for c in groups[name]]))
                same=(m["configs"]==expected and m["ebno_db"]==ebno_by_group[name]
                      and m["packets"]==PACKETS and m["warmups"]==WARMUPS
                      and m["source_hash"]==fingerprint()
                      and m.get("torch")==torch.__version__ and m.get("sionna")==sionna.__version__)
                if not same: raise RuntimeError("Saved group differs from these settings/source/environment. Change RUN_LABEL to start a separate run.")
                print(f"{name}: reusing completed, matching run.")
            else:
                if folder.exists():
                    folder=OUTPUT_ROOT/f"{name}_retry_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%f')}"
                    print("An incomplete attempt is preserved; restarting this group in",folder)
                GROUP_PATHS[name]=str(folder)
                INDEX_PATH.write_text(json.dumps(GROUP_PATHS,indent=2))
                print(f"Running {name}: {len(groups[name])} configurations × {len(ebno_by_group[name])} SNRs × {PACKETS} batches",flush=True)
                run_group(groups[name],ebno_by_group[name],folder,validation,registry,packets=PACKETS,warmups=WARMUPS)
            GROUP_PATHS[name]=str(folder)
            INDEX_PATH.write_text(json.dumps(GROUP_PATHS,indent=2))
        report_group(name,folder)
        STATUS[name]="complete" if MODE=="run" else "displayed saved results"
    except Exception as error:
        STATUS[name]=f"FAILED: {type(error).__name__}: {error}"
        traceback.print_exc()
        print("This group failed. Its partial files are preserved; other experiment cells can continue. Inspect the final status table.")
    (OUTPUT_ROOT/"status.json").write_text(json.dumps(STATUS,indent=2))


## Experiment 1. Conventional PHY: uncoded versus LDPC

Establish the conventional reference and quantify the reliability/processing-cost tradeoff.


In [ ]:
execute_and_report('01_conventional_coding')


## Experiment 2. Conventional PHY versus full-band GGH

Measure the cost of introducing GGH under the same transmit-power normalization.


In [ ]:
execute_and_report('02_full_ggh')


## Experiment 3. Full-band GGH: uncoded versus LDPC

Test whether the hard-to-soft LDPC baseline helps; compare BER even if BLER saturates.


In [ ]:
execute_and_report('03_ggh_coding')


## Experiment 4. Full-band versus sub-band GGH

Compare 128×4 and 64×8 tiles at n=512, with and without LDPC.


In [ ]:
execute_and_report('04_localization')


## Experiment 5. Receiver processing

Compare ZF, LMMSE and MMSE-SIC. Ciphertext SIC uses continuous cancellation, not QAM slicing.


In [ ]:
execute_and_report('05_receiver')


## Experiment 6. Channel-state information

Compare perfect, LS-nearest-neighbor, LS-linear and controlled-NMSE estimates.


In [ ]:
execute_and_report('06_csi')


## Experiment 7a. User-count sensitivity

Vary K while holding the other reference settings fixed.


In [ ]:
execute_and_report('07a_users')


## Experiment 7b. Base-station antenna sensitivity

Vary M separately from K.


In [ ]:
execute_and_report('07b_antennas')


## Experiment 8. Modulation sensitivity

Compare QPSK, 16-QAM and 64-QAM.


In [ ]:
execute_and_report('08_modulation')


## Experiment 9. Channel-model sensitivity

Compare UMi, UMa and RMa under the documented normalized-channel assumptions.


In [ ]:
execute_and_report('09_channel')


## Experiment 10. System-level overhead

Compare all six schemes. Inspect latency, goodput, power expansion and tensor storage together.


In [ ]:
execute_and_report('10_overhead')


## 6. Combined results and completion check

Review this table before interpreting the run. A failed or missing group is not a completed experiment. The highest-SNR table is a compact overview, not a replacement for the full curves. All successful group summaries are combined into one CSV with an explicit experiment-group column.


In [ ]:
status_table = pd.DataFrame([{"group":name,"status":STATUS.get(name,"not run")} for name in groups])
display(status_table)
status_table.to_csv(OUTPUT_ROOT/"experiment_status.csv",index=False)
if REPORTS:
    combined = pd.concat([df.assign(experiment=name) for name,df in REPORTS.items()],ignore_index=True)
    combined.to_csv(OUTPUT_ROOT/"combined_summary.csv",index=False)
    latest = combined[combined.ebno_db == combined.groupby("experiment").ebno_db.transform("max")]
    display(latest[["experiment","configuration","ebno_db","ber","bler","goodput","total_ms"]].round(5))
    print("Groups displayed:",len(REPORTS),"of",len(groups))
print("Results and report directory:", OUTPUT_ROOT)


## 7. Archive and download

This archives everything generated by this companion run: raw files, manifests, key metadata, validation, tables and figures. In view-only mode, the archive contains new reports and their provenance; original saved inputs stay where they were. Choose `DOWNLOAD_ZIP=True` to start a Colab download. Save the archive before deleting a temporary runtime. Nothing is pushed to GitHub automatically.


In [ ]:
import shutil
DOWNLOAD_ZIP = False
archive_path = Path(shutil.make_archive(str(OUTPUT_ROOT)+"_export", "zip", root_dir=OUTPUT_ROOT))
print("Archive:", archive_path)
if IN_COLAB:
    from IPython.display import HTML
    if str(archive_path).startswith("/content/"):
        from html import escape
        from urllib.parse import quote
        relative = archive_path.relative_to("/content").as_posix()
        display(HTML(f'<a href="/files/{quote(relative)}" download>{escape(archive_path.name)}</a>'))
    if DOWNLOAD_ZIP:
        from google.colab import files
        files.download(str(archive_path))
else:
    from IPython.display import FileLink
    display(FileLink(str(archive_path)))


## What to do after the pilot

Keep raw observations and manifests. Inspect paired baseline/encryption comparisons before making a reliability claim. Where all GGH curves saturate, investigate ciphertext expansion, sigma, Babai errors and U-inverse error propagation; do not infer that more samples alone will produce a gain. For measurable curves, refine the SNR grid and increase independent batches near the target BLER. Repeat with additional seeds as a separate study. CPU and GPU timings are different measurements; never combine them silently.
